In [1]:
import os
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from datasets import load_from_disk

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['figure.dpi'] = 300

In [3]:
corpus_name = 'hst_corpus_90m' # 'hst_corpus_80m'
tokenized_data_str = f'{corpus_name}_32_None_None_None_None_None_shifted_log_knn_10' # 'hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10', 'hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10'
tokenized_data_folder_path = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/{corpus_name}/{tokenized_data_str}.dataset' # f'/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/{tokenized_data_str}.dataset'
silver_folder_path = '/nfs/team361/sb75/DATASETS/silver/'
tokenizer_folder_path = '/nfs/team361/sb75/DATASETS/tokenizer'

In [4]:
# Load dataset
dataset = load_from_disk(tokenized_data_folder_path)
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 88626282
})

In [5]:
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 88626282
})

## Dataset-based Cell Sampling

In [6]:
# Retrieve dataset IDs
dataset_ids = [cell_id.split('_')[0] for cell_id in list(dataset['cell_id'])]

In [7]:
dataset_ids_set = set(dataset_ids)

In [8]:
random.seed(0)
sample_size = int(len(dataset_ids_set) * 0.1)  # 10% of elements
dataset_ids_sample = random.sample(dataset_ids_set, sample_size)
dataset_ids_sample_set = set(dataset_ids_sample)

/tmp/ipykernel_710255/102669146.py:3: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  dataset_ids_sample = random.sample(dataset_ids_set, sample_size)


In [ ]:
num_workers = 12
sampled_dataset_ids = []

i = 0
sample_ratios = [0.01, 0.1, 0.25, 0.5]
for dataset_id in random.sample(dataset_ids_set, len(dataset_ids_set)):
    sample_ratio = sample_ratios[i]
    sampled_dataset_ids.append(dataset_id)
    sampled_dataset_ids_set = set(sampled_dataset_ids)

    sampled_dataset = dataset.filter(
        lambda ids: [i.split('_')[0] in sampled_dataset_ids_set for i in ids],
        input_columns="cell_id",
        batched=True,
        batch_size=10000,
        num_proc=num_workers,
        desc="Filtering by dataset_id"
    )
    
    n_sampled_cells = len(sampled_dataset)
    
    print(f'Intermediate number of cells {n_sampled_cells}')
    if n_sampled_cells > int(sample_ratio * len(dataset)):
        sampled_dataset = sampled_dataset.select(range(int(sample_ratio * len(dataset))))
        print(f'Final number of cells for sample ratio {sample_ratio}: {n_sampled_cells}')
        i += 1
        print('Saving sampled dataset...')
        sampled_dataset.save_to_disk(tokenized_data_folder_path.replace('hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10', f'hst_corpus_90m_{sample_ratio}_sample_32_None_None_None_None_None_shifted_log_knn_10'))

/tmp/ipykernel_710255/2601180567.py:6: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  for dataset_id in random.sample(dataset_ids_set, len(dataset_ids_set)):
Filtering by dataset_id (num_proc=12): 100%|██████████| 88626282/88626282 [00:19<00:00, 4621735.58 examples/s]


Intermediate number of cells 1434182
Final number of cells for sample ratio 0.01: 1434182
Saving sampled dataset...


Filtering by dataset_id (num_proc=12): 100%|██████████| 88626282/88626282 [00:27<00:00, 3256603.09 examples/s]


Intermediate number of cells 1582398


Filtering by dataset_id (num_proc=12): 100%|██████████| 88626282/88626282 [00:19<00:00, 4514146.72 examples/s]


Intermediate number of cells 2119900


... (13 outputs left out)

Intermediate number of cells 6192938


Filtering by dataset_id (num_proc=12): 100%|██████████| 88626282/88626282 [00:19<00:00, 4506771.97 examples/s]


Intermediate number of cells 6427922


Filtering by dataset_id (num_proc=12):  99%|█████████▉| 87934188/88626282 [00:18<00:00, 4618255.85 examples/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Saving the dataset (157/2000 shards):   8%|▊         | 697824/8862628 [18:43<4:08:38, 547.31 examples/s] 

In [ ]:
sampled_dataset = dataset.filter(
    lambda ids: [i.split('_')[0] in dataset_ids_sample_set for i in ids],
    input_columns="cell_id",
    batched=True,
    batch_size=10000,
    num_proc=num_workers,
    desc="Filtering by dataset_id"
)

In [ ]:
sampled_dataset

In [ ]:
num_workers = 12

def keep_row(x):
    return torch.isin(torch.tensor(int(x["cell_id"][0].split('_')[0])), dataset_ids_tensor)

sampled_dataset = dataset.filter(
    keep_row,
    batched=True,
    batch_size=10000, 
    num_proc=num_workers)
print(sampled_dataset)

## Stratified Cell Sampling

In [ ]:
# Define sample config
n_samples = 30
sample_type = 'frac_per_bin' # 'cells_per_bin', 'frac_per_bin', 'mixed'
avg_cells_per_bin = 50
sampled_cells_per_bin = 1
sample_frac = 0.1

In [ ]:
# Retrieve cell IDs
dataset_cell_ids = dataset['cell_id']

In [ ]:
# Create cell ID dict
cell_id_dict = {v: i for i, v in enumerate(dataset_cell_ids)} # for speedup

In [ ]:
all_epochs_indices = [[] for i in range(n_samples)]

for subdir, _, files in os.walk(silver_folder_path + corpus_name):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print('====================================================================================================')
            print(f'Processing adata with file path {file_path}...')

            # Retrieve x and y bins
            print('Creating bins...')
            n_bins = int(np.round(len(adata) / avg_cells_per_bin, decimals=0))
            x_bins = np.linspace(adata.obsm['spatial'][:, 0].min(), adata.obsm['spatial'][:, 0].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
            y_bins = np.linspace(adata.obsm['spatial'][:, 1].min(), adata.obsm['spatial'][:, 1].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)

            # Assign each point to a bin
            adata.obs['x_bin'] = pd.cut(adata.obsm['spatial'][:, 0], bins=x_bins, labels=False, include_lowest=True)
            adata.obs['y_bin'] = pd.cut(adata.obsm['spatial'][:, 1], bins=y_bins, labels=False, include_lowest=True)

            print('Sampling cells...')
            indices = [] 
            for i in range(n_samples):
                if sample_type == 'cells_per_bin':
                    sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                elif sample_type == 'frac_per_bin':
                    sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                elif sample_type == 'mixed':
                    # First, sample 1 point from each group
                    min_sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                    
                    # Then, sample the remaining points based on sample_frac
                    remaining_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                    
                    # Combine both sampled DataFrames
                    sampled_df = pd.concat([min_sampled_df, remaining_df]).drop_duplicates()
                
                sampled_df = sampled_df.reset_index(drop=True)
            
                #adata.obs['sampled_flag'] = adata.obs['cell_id'].apply(lambda x: x in sampled_df['cell_id'].values.tolist())
    
                indices.append([cell_id_dict.get(item, None) for item in sampled_df['cell_id'].values.tolist()])
                del(sampled_df)
            
                #sc.pl.spatial(adata,
                #              color='sampled_flag',
                #              spot_size=spot_size)

            del(adata)
            for i, epoch_indices in enumerate(indices):
                all_epochs_indices[i].extend(epoch_indices)

            print('Saving results to disk...')
            with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_split_{n_samples}.pkl', 'wb') as f:
                pickle.dump(all_epochs_indices, f)

all_epochs_n_nonzero_tokens = []

for epoch_indices in all_epochs_indices:
    epoch_data = dataset.select(epoch_indices)
    epoch_n_nonzero_tokens = epoch_data['n_nonzero_tokens']
    all_epochs_n_nonzero_tokens.append(epoch_n_nonzero_tokens)

with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{n_samples}.pkl', 'wb') as f:
    pickle.dump(all_epochs_n_nonzero_tokens, f)

In [ ]:
# Sample batches
for file_path_sample_ratio in [0.01, 0.1, 0.25, 0.5]:
    n_samples = int(30 / file_path_sample_ratio)
    file_paths = []
    
    # for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_70m'):
    for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_80m'):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                file_paths.append(file_path)
    
    file_path_sample_n = max(1, int(len(file_paths) * file_path_sample_ratio))
    sampled_file_paths = random.sample(file_paths, file_path_sample_n)
    
    all_epochs_indices = [[] for i in range(n_samples)]
    
    # for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_70m'):
    for subdir, _, files in os.walk(silver_folder_path + 'hst_corpus_80m'):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                if file_path in sampled_file_paths:
                    adata = sc.read_h5ad(file_path)
                    print('====================================================================================================')
                    print(f'Processing adata with file path {file_path}...')
        
                    # Retrieve x and y bins
                    print('Creating bins...')
                    n_bins = int(np.round(len(adata) / avg_cells_per_bin, decimals=0))
                    x_bins = np.linspace(adata.obsm['spatial'][:, 0].min(), adata.obsm['spatial'][:, 0].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
                    y_bins = np.linspace(adata.obsm['spatial'][:, 1].min(), adata.obsm['spatial'][:, 1].max(), int(np.round(np.sqrt(n_bins), decimals=0)) + 1)
        
                    # Assign each point to a bin
                    adata.obs['x_bin'] = pd.cut(adata.obsm['spatial'][:, 0], bins=x_bins, labels=False, include_lowest=True)
                    adata.obs['y_bin'] = pd.cut(adata.obsm['spatial'][:, 1], bins=y_bins, labels=False, include_lowest=True)
        
                    print('Sampling cells...')
                    indices = [] 
                    for i in range(n_samples):
                        if sample_type == 'cells_per_bin':
                            sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                        elif sample_type == 'frac_per_bin':
                            sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                        elif sample_type == 'mixed':
                            # First, sample 1 point from each group
                            min_sampled_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(n=sampled_cells_per_bin))
                            
                            # Then, sample the remaining points based on sample_frac
                            remaining_df = adata.obs.groupby(['x_bin', 'y_bin']).apply(lambda group: group.sample(frac=sample_frac))
                            
                            # Combine both sampled DataFrames
                            sampled_df = pd.concat([min_sampled_df, remaining_df]).drop_duplicates()
                        
                        sampled_df = sampled_df.reset_index(drop=True)
                    
                        #adata.obs['sampled_flag'] = adata.obs['cell_id'].apply(lambda x: x in sampled_df['cell_id'].values.tolist())
            
                        indices.append([cell_id_dict.get(item, None) for item in sampled_df['cell_id'].values.tolist()])
                        del(sampled_df)
                    
                        #sc.pl.spatial(adata,
                        #              color='sampled_flag',
                        #              spot_size=spot_size)
        
                    del(adata)
                    for i, epoch_indices in enumerate(indices):
                        all_epochs_indices[i].extend(epoch_indices)
        
                    print('Saving results to disk...')
                    with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
                        pickle.dump(all_epochs_indices, f)
        
    all_epochs_n_nonzero_tokens = []
    
    for epoch_indices in all_epochs_indices:
        epoch_data = dataset.select(epoch_indices)
        epoch_n_nonzero_tokens = epoch_data['n_nonzero_tokens']
        all_epochs_n_nonzero_tokens.append(epoch_n_nonzero_tokens)
    
    with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
        pickle.dump(all_epochs_n_nonzero_tokens, f)

In [ ]:
file_path_sample_ratio = 0.25
n_samples = int(1 / file_path_sample_ratio) * 10

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_full = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_full = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_30.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_30 = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_30.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_30 = pickle.load(file)

In [ ]:
len(precomputed_epoch_split_subsample_full)

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full)

In [ ]:
len(precomputed_epoch_split_subsample_full[0])

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full[0])

In [ ]:
len(precomputed_epoch_split_subsample_full[-1])

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full[-1])

In [ ]:
len(precomputed_epoch_split_subsample_30[0])

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_30[0])

In [ ]:
precomputed_epoch_split_subsample_full_new = precomputed_epoch_split_subsample_30 + precomputed_epoch_split_subsample_full[30:]
precomputed_epoch_n_nonzero_tokens_subsample_full_new = precomputed_epoch_n_nonzero_tokens_subsample_30 + precomputed_epoch_n_nonzero_tokens_subsample_full[30:]

In [ ]:
with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
    pickle.dump(precomputed_epoch_split_subsample_full_new, f)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'wb') as f:
    pickle.dump(precomputed_epoch_n_nonzero_tokens_subsample_full_new, f)

In [ ]:
with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_split_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_split_subsample_full_new = pickle.load(file)

with open(f'/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/{tokenized_data_str}_precomputed_epoch_n_nonzero_tokens_{int(file_path_sample_ratio * 100)}_pct_subsample_{n_samples}.pkl', 'rb') as file:
    precomputed_epoch_n_nonzero_tokens_subsample_full_new = pickle.load(file)

In [ ]:
len(precomputed_epoch_split_subsample_full_new)

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full_new)

In [ ]:
len(precomputed_epoch_split_subsample_full_new[-1])

In [ ]:
len(precomputed_epoch_n_nonzero_tokens_subsample_full_new[-1])